#### 适用范围：相关性分析
例如：天气预报，销售额分析  
**核心：计算变量之间的关联度**
- 应用：仿照topsis构造理想最优值，计算每个指标的关联系数，与理想最优值关联度最大的方案为最优

### 数据正向化，并统一量纲

In [1]:
import numpy as np

# ==================== 正向化函数 ====================
def min_to_max(x):
    """极小型 -> 极大型"""
    return np.max(x) - x

def best_to_max(x, best):
    """中间型 -> 极大型"""
    M = np.max(np.abs(x - best))
    if M == 0:
        return np.ones_like(x, dtype=float)
    return 1 - np.abs(x - best) / M

def fuzzy_interval_membership(x, lower, upper):
    """区间型 -> 极大型"""
    left_dist = np.maximum(lower - x, 0)
    right_dist = np.maximum(x - upper, 0)
    dist = left_dist + right_dist
    M = np.max(dist)
    if M == 0:
        return np.ones_like(x, dtype=float)
    return 1 - dist / M


# ==================== 灰关联数据预处理 ====================
def preprocess_for_gra(A, kinds, best_dict=None, interval_dict=None):
    """
    对原始数据进行正向化和无量纲化（均值化）
    
    参数:
        A: np.ndarray, 原始数据矩阵，行=样本，列=指标
        kinds: list, 指标类型，1=极大型, 2=极小型, 3=中间型, 4=区间型
        best_dict: dict, 中间型指标的最优值，如 {2: 25}
        interval_dict: dict, 区间型指标的区间，如 {3: (10, 20)}
    
    返回:
        X_normalized: np.ndarray, 正向化+均值化后的矩阵
    """
    if best_dict is None:
        best_dict = {}
    if interval_dict is None:
        interval_dict = {}
    
    n, m = A.shape
    X_positive = np.zeros((n, m))
    
    for i in range(m):
        col = A[:, i].astype(float)
        kind = kinds[i]
        
        if kind == 1:        # 极大型，不处理
            X_positive[:, i] = col
        elif kind == 2:      # 极小型
            X_positive[:, i] = min_to_max(col)
        elif kind == 3:      # 中间型
            if i not in best_dict:
                raise ValueError(f"第 {i+1} 列是中间型指标，请在 best_dict 中提供最优值")
            X_positive[:, i] = best_to_max(col, best_dict[i])
        elif kind == 4:      # 区间型
            if i not in interval_dict:
                raise ValueError(f"第 {i+1} 列是区间型指标，请在 interval_dict 中提供区间")
            lower, upper = interval_dict[i]
            X_positive[:, i] = fuzzy_interval_membership(col, lower, upper)
        else:
            raise ValueError(f"第 {i+1} 列指标类型 {kind} 无效，请使用 1-4")
    
    # 均值化（灰色关联分析常用）
    X_mean = X_positive.mean(axis=0)
    X_mean[X_mean == 0] = 1e-6  # 防止除以0
    X_normalized = X_positive / X_mean
    
    return X_normalized


# ==================== 使用示例 ====================
if __name__ == "__main__":
    # 手动输入数据（行=样本，列=指标）
    A = np.array([[90, 4.0, 35, 15],
                  [85, 3.0, 50, 21],
                  [92, 4.5, 40, 19],
                  [88, 3.5, 38, 17]])

    # 指标类型：1=极大, 2=极小, 3=中间, 4=区间
    kinds = [1, 2, 3, 4]

    # 参数设定
    best_dict = {2: 40}           # 第3列(索引2)是中间型，最优值40
    interval_dict = {3: (15, 20)}  # 第4列(索引3)是区间型，区间[15,20]

    # 预处理
    result = preprocess_for_gra(A, kinds, best_dict, interval_dict)

    print("原始数据：\n", A)
    print("\n预处理后数据（正向化+均值化）：\n", result)

原始数据：
 [[90.   4.  35.  15. ]
 [85.   3.  50.  21. ]
 [92.   4.5 40.  19. ]
 [88.   3.5 38.  17. ]]

预处理后数据（正向化+均值化）：
 [[1.01408451 0.66666667 0.86956522 1.33333333]
 [0.95774648 2.         0.         0.        ]
 [1.03661972 0.         1.73913043 1.33333333]
 [0.9915493  1.33333333 1.39130435 1.33333333]]


### 计算关联度
- 公式：  
$\xi_{ij} = \frac{a+\rho b}{X_{ij}+\rho b}$  
$r_i = \sum_{i=1}^{n}\xi_{ij}$  
其中，$X_{ij}$ 为被评价行与参照行的差，a 为 $X_{ij}$ 的两级最小差，b 为 $X_{ij}$ 的两级最大差，$r_{j}$ 为关联度
   
在评价类问题中，我们将使用与topsis法相似的方法计算出一个理想最优解，将其作为参照

In [ ]:
def gray_relational_analysis_weighted(X, weights, rho=0.5):
    """
    加权灰色关联分析
    
    参数:
        X: np.ndarray, 预处理后的数据
        weights: list/np.ndarray, 各指标权重，长度等于指标数
        rho: 分辨系数，默认 0.5
    
    返回:
        gamma: np.ndarray, 加权灰色关联度
        xi: np.ndarray, 关联系数矩阵
    """
    n, m = X.shape
    if weights is None:
        weights = np.ones(m) / m
    else:
        if len(weights) != m:
            raise ValueError(f"权重长度 {len(weights)} 与指标数 {m} 不一致")
        weights = np.array(weights) / np.sum(weights)  
    
    # 参考序列
    ref_seq = np.max(X, axis=0)
    
    # 绝对差值
    delta = np.abs(X - ref_seq)
    
    # 两极差
    delta_min = np.min(delta)
    delta_max = np.max(delta)
    
    # 关联系数
    xi = (delta_min + rho * delta_max) / (delta + rho * delta_max)
    
    # 加权关联度
    r = np.sum(xi * weights, axis=1)
    
    return r, xi
